# 국내·해외 × 품목명(`train_prdlst_nm`) 양성률

- 데이터: `리스테리아ML/data/listeria_df_joined.parquet` + `리스테리아ML/data/X.parquet` (`src_route`)
- 국내·해외: 로마자로 적힌 한국 주소 78행을 overseas → domestic 으로 옮긴 `src_route` (`EXP/1/1_데이터정제.ipynb`)
- **양성률 = 칸의 검출 수 ÷ 전체 검출 수 (%)**. 품목명이 없는 행은 '(품목명 없음)'
- 정렬: 전체 검출 수가 많은 순

In [1]:
import pandas as pd
pd.set_option("display.max_rows", 200)
DATA = r"C:\cheon\cheon_wokespace\listeria2026\리스테리아ML\data"
d = pd.read_parquet(f"{DATA}/listeria_df_joined.parquet", columns=["lims_data_key", "train_prdlst_nm", "judge_result"])
x = pd.read_parquet(f"{DATA}/X.parquet", columns=["lims_data_key", "src_route"])
d = d.merge(x, on="lims_data_key", how="left", validate="one_to_one")
assert d["src_route"].notna().all()
d["y"] = (d["judge_result"] == "검출").astype(int)
d["국내외"] = d["src_route"].astype("string").map({"domestic": "국내", "overseas": "해외"})
d["품목명"] = d["train_prdlst_nm"].astype("string").fillna("(품목명 없음)")
N_POS = int(d["y"].sum())
print(f"전체 {len(d):,}행 · 전체 양성 {N_POS:,}건 · 품목명 {d['품목명'].nunique()}개")
print(d["국내외"].value_counts())

전체 676,537행 · 전체 양성 1,969건 · 품목명 53개
국내외
국내    628973
해외     47564
Name: count, dtype: int64


## 양성률 표 (분모 = 전체 양성)

검출 수, 행 수, 칸 안 검출률(분모 = 칸의 행 수)을 같은 표에 붙였다.

In [2]:
pos = pd.crosstab(d["품목명"], d["국내외"], values=d["y"], aggfunc="sum", margins=True, margins_name="합계").fillna(0).astype(int)
rows = pd.crosstab(d["품목명"], d["국내외"], margins=True, margins_name="합계")
cols = ["국내", "해외", "합계"]
t = pd.concat({"양성률(%)": (pos[cols] / N_POS * 100).round(2),
               "검출 수": pos[cols],
               "행 수": rows[cols],
               "칸 안 검출률(%)": (pos[cols] / rows[cols].astype(float).replace(0, float("nan")) * 100).round(4)}, axis=1)
body = t.drop(index="합계").sort_values([("검출 수", "합계"), ("행 수", "합계")], ascending=False)
t = pd.concat([body, t.loc[["합계"]]])
t

양성률(%)                검출 수                행 수                 \
국내외                 국내    해외      합계    국내   해외    합계      국내     해외      합계   
품목명                                                                            
기타기준규격외          40.17  0.00   40.17   791    0   791  291799     14  291813   
정액,수정란,종란        32.50  0.00   32.50   640    0   640   28651      0   28651   
식육가공품 및 포장육      14.93  1.47   16.40   294   29   323   95216   5829  101045   
수산물               0.15  2.18    2.34     3   43    46     479   7994    8473   
즉석식품류             2.29  0.00    2.29    45    0    45   35322      0   35322   
유가공품류             0.30  1.73    2.03     6   34    40   78292  28730  107022   
수산가공식품류           1.47  0.00    1.47    29    0    29    4705    210    4915   
축산물               0.46  0.96    1.42     9   19    28     966    765    1731   
버섯류               0.61  0.00    0.61    12    0    12      92      0      92   
기타식품류             0.46  0.00    0.46     9    0     9    7381      3    7384   
기타식물류             0.10  0.00    0.10     2    0     2      43      0      43   
음료류               0.05  0.00    0.05     1    0     1    6279    119    6398   
과자류, 빵류 또는 떡류     0.05  0.00    0.05     1    0     1    3919     17    3936   
농산가공식품류           0.05  0.00    0.05     1    0     1    2908     15    2923   
견과 종실류            0.05  0.00    0.05     1    0     1     292      0     292   
알가공품류             0.00  0.00    0.00     0    0     0   28465    927   29392   
빙과류               0.00  0.00    0.00     0    0     0   19816   2390   22206   
조리식품 등            0.00  0.00    0.00     0    0     0    9590      0    9590   
조미식품              0.00  0.00    0.00     0    0     0    4165      0    4165   
절임류 또는 조림류        0.00  0.00    0.00     0    0     0    2804      0    2804   
특수영양식품            0.00  0.00    0.00     0    0     0    1613    527    2140   
채소류               0.00  0.00    0.00     0    0     0     904      0     904   
코코아가공품류 또는 초콜릿류   0.00  0.00    0.00     0    0     0     720     15     735   
과일류               0.00  0.00    0.00     0    0     0     702      0     702   
두부류 또는 묵류         0.00  0.00    0.00     0    0     0     489      0     489   
장류                0.00  0.00    0.00     0    0     0     430      0     430   
면류                0.00  0.00    0.00     0    0     0     408      3     411   
식용유지류             0.00  0.00    0.00     0    0     0     368      1     369   
기타제조가공식품류(자가규격)   0.00  0.00    0.00     0    0     0     342      0     342   
당류                0.00  0.00    0.00     0    0     0     295      1     296   
기능성 원료            0.00  0.00    0.00     0    0     0     271      0     271   
동물성가공식품류          0.00  0.00    0.00     0    0     0     189      0     189   
기타                0.00  0.00    0.00     0    0     0     166      4     170   
개별인정원료            0.00  0.00    0.00     0    0     0     163      0     163   
잼류                0.00  0.00    0.00     0    0     0     148      0     148   
특수의료용도식품          0.00  0.00    0.00     0    0     0      82      0      82   
주류                0.00  0.00    0.00     0    0     0      74      0      74   
식품첨가물             0.00  0.00    0.00     0    0     0      70      0      70   
혼합제제류             0.00  0.00    0.00     0    0     0      68      0      68   
벌꿀 및 화분가공품류       0.00  0.00    0.00     0    0     0      56      0      56   
곡류                0.00  0.00    0.00     0    0     0      52      0      52   
접객용 음용수           0.00  0.00    0.00     0    0     0      50      0      50   
수산가공              0.00  0.00    0.00     0    0     0      40      0      40   
환경검체              0.00  0.00    0.00     0    0     0      35      0      35   
환경오염도조사용          0.00  0.00    0.00     0    0     0      14      0      14   
영양성분              0.00  0.00    0.00     0    0     0      12      0      12   
농산물 종자류           0.00  0.00    0.00     0    0     0       8      0       8   
두류              

## 검출이 있는 품목만

In [3]:
t[t[("검출 수", "합계")] > 0]

양성률(%)                검출 수                행 수                 \
국내외               국내    해외      합계    국내   해외    합계      국내     해외      합계   
품목명                                                                          
기타기준규격외        40.17  0.00   40.17   791    0   791  291799     14  291813   
정액,수정란,종란      32.50  0.00   32.50   640    0   640   28651      0   28651   
식육가공품 및 포장육    14.93  1.47   16.40   294   29   323   95216   5829  101045   
수산물             0.15  2.18    2.34     3   43    46     479   7994    8473   
즉석식품류           2.29  0.00    2.29    45    0    45   35322      0   35322   
유가공품류           0.30  1.73    2.03     6   34    40   78292  28730  107022   
수산가공식품류         1.47  0.00    1.47    29    0    29    4705    210    4915   
축산물             0.46  0.96    1.42     9   19    28     966    765    1731   
버섯류             0.61  0.00    0.61    12    0    12      92      0      92   
기타식품류           0.46  0.00    0.46     9    0     9    7381      3    7384   
기타식물류           0.10  0.00    0.10     2    0     2      43      0      43   
음료류             0.05  0.00    0.05     1    0     1    6279    119    6398   
과자류, 빵류 또는 떡류   0.05  0.00    0.05     1    0     1    3919     17    3936   
농산가공식품류         0.05  0.00    0.05     1    0     1    2908     15    2923   
견과 종실류          0.05  0.00    0.05     1    0     1     292      0     292   
합계             93.65  6.35  100.00  1844  125  1969  628973  47564  676537   

              칸 안 검출률(%)                   
국내외                   국내      해외       합계  
품목명                                        
기타기준규격외           0.2711  0.0000   0.2711  
정액,수정란,종란         2.2338     NaN   2.2338  
식육가공품 및 포장육       0.3088  0.4975   0.3197  
수산물               0.6263  0.5379   0.5429  
즉석식품류             0.1274     NaN   0.1274  
유가공품류             0.0077  0.1183   0.0374  
수산가공식품류           0.6164  0.0000   0.5900  
축산물               0.9317  2.4837   1.6176  
버섯류              13.0435     NaN  13.0435  
기타식품류             0.1219  0.0000   0.1219  
기타식물류             4.6512     NaN   4.6512  
음료류               0.0159  0.0000   0.0156  
과자류, 빵류 또는 떡류     0.0255  0.0000   0.0254  
농산가공식품류           0.0344  0.0000   0.0342  
견과 종실류            0.3425     NaN   0.3425  
합계                0.2932  0.2628   0.2910